In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ================================================================
# DEEPSEEK-CODER + PRIMEVUL (FINAL WORKING VERSION)
# ================================================================

import torch
import re
import json
import os
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef

# ================================================================
# CONFIG
# ================================================================
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"
MAX_INPUT_TOKENS = 1200
MAX_NEW_TOKENS = 5

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

# ================================================================
# LOAD DATASET
# ================================================================
DATASET_PATH = None

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if "test" in f and f.endswith(".jsonl"):
            DATASET_PATH = os.path.join(root, f)
            break
    if DATASET_PATH:
        break

print(f"✅ Dataset: {DATASET_PATH}")

dataset = [json.loads(line) for line in open(DATASET_PATH)]
print(f"✅ Samples: {len(dataset)}")

# ================================================================
# LOAD MODEL
# ================================================================
print("\n⏳ Loading DeepSeek-Coder...\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

# ================================================================
# PREPROCESS
# ================================================================
def preprocess(code):
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'//.*', '', code)
    code = re.sub(r'#.*', '', code)
    return code.strip()

def truncate_code(code):
    tokens = tokenizer.encode(code, truncation=True, max_length=MAX_INPUT_TOKENS)
    return tokenizer.decode(tokens)

# ================================================================
# PROMPT (OPTIMIZED FOR CODE MODELS)
# ================================================================
def build_prompt(code):
    code = preprocess(code)
    code = truncate_code(code)

    return f"""### Instruction:
Detect whether the following C/C++ code contains a security vulnerability.

Answer only Yes or No.

### Code:
{code}

### Answer:"""

# ================================================================
# PREDICTION
# ================================================================
@torch.no_grad()
def predict(code):
    prompt = build_prompt(code)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

    gen = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen).lower()

    if "yes" in text:
        return 1
    elif "no" in text:
        return 0
    else:
        return 0

# ================================================================
# INFERENCE
# ================================================================
y_true, y_pred = [], []

print("\n🚀 Running DeepSeek-Coder inference...\n")

for sample in tqdm(dataset):
    code = sample["func"]
    label = sample["target"]

    pred = predict(code)

    y_true.append(label)
    y_pred.append(pred)

    torch.cuda.empty_cache()

# ================================================================
# METRICS
# ================================================================
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
mcc  = matthews_corrcoef(y_true, y_pred)
cm   = confusion_matrix(y_true, y_pred)

# ================================================================
# RESULTS
# ================================================================
print("\n🚀 FINAL RESULTS (DEEPSEEK-CODER)")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"MCC       : {mcc:.4f}")

print("\nConfusion Matrix:")
print(cm)

# ================================================================
# SAVE
# ================================================================
df = pd.DataFrame({
    "true": y_true,
    "pred": y_pred
})

output_path = OUT_DIR / "deepseek_coder_results.csv"
df.to_csv(output_path, index=False)

print(f"\n✅ Results saved at: {output_path}")

✅ Dataset: /kaggle/input/datasets/nikunjnawal009/deepseek-primevul/primevul_test_paired.jsonl
✅ Samples: 870

⏳ Loading DeepSeek-Coder...



config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]


🚀 Running DeepSeek-Coder inference...



100%|██████████| 870/870 [04:09<00:00,  3.48it/s]


🚀 FINAL RESULTS (DEEPSEEK-CODER)
Accuracy  : 0.5023
Precision : 0.5020
Recall    : 0.5678
F1 Score  : 0.5329
MCC       : 0.0046

Confusion Matrix:
[[190 245]
 [188 247]]

✅ Results saved at: /kaggle/working/deepseek_coder_results.csv
